# **Trial outcome derivation**

### *Clinical Trial CTN-0051: Extended-Release Naltroxone vs. Buprenorphine for Opioid Treatment*


In this notebook, we extract the dropout and relapse data that classifies the patients.

In [2]:
import pandas as pd
import numpy as np

c:\Users\Miki\anaconda3\Lib\site-packages\pandas\core\computation\expressions.py:22: UserWarning: Pandas requires version '2.10.2' or newer of 'numexpr' (version '2.10.1' currently installed).
  from pandas.core.computation.check import NUMEXPR_INSTALLED


In [4]:
# ── Load XRP ──────────────────────────────────────────────────────────────────
xrp = pd.read_csv("../raw-data/XRP.csv")

xrp.head(6)

,PROT,PATID,SITE,RANDDT,PROTSEG,VISNO,XRPASMDT,XRRELAPS,XRRLPCRT,XRPCOMM
0,51,948442,NaN,0,B,24,189,0,NaN,NaN
1,51,948442,NaN,0,B,20,137,0,NaN,NaN
2,51,948442,NaN,0,B,18,124,0,NaN,NaN
3,51,948442,NaN,0,B,16,109,0,NaN,NaN
4,51,948442,NaN,0,B,12,89,0,NaN,NaN
5,51,948442,NaN,0,B,10,70,0,NaN,NaN


In [5]:


# ── Convert VISNO to integer ──────────────────────────────────────────────────
xrp['visno_int'] = pd.to_numeric(xrp['VISNO'], errors='coerce')

# ── Per-patient summary from XRP ──────────────────────────────────────────────
xrp_agg = (
    xrp.groupby('PATID')
    .agg(
        n_xrp_visits  = ('visno_int', 'count'),
        last_xrp_week = ('visno_int', 'max'),
        had_relapse   = ('XRRELAPS',  'max'),
    )
    .reset_index()
)

# ── First relapse week ────────────────────────────────────────────────────────
first_relapse = (
    xrp[xrp['XRRELAPS'] == 1]
    .groupby('PATID')['visno_int']
    .min()
    .reset_index()
    .rename(columns={'visno_int': 'first_relapse_week'})
)
xrp_agg = xrp_agg.merge(first_relapse, on='PATID', how='left')

# ── Derive retention tier ─────────────────────────────────────────────────────
def assign_tier(row):
    if row['n_xrp_visits'] == 0 or pd.isna(row['last_xrp_week']):
        return 0   # never engaged
    w = row['last_xrp_week']
    if w >= 24: return 4   # completed
    if w >= 16: return 3   # late dropout
    if w >= 8:  return 2   # mid dropout
    return 1               # early dropout

xrp_agg['retention_tier'] = xrp_agg.apply(assign_tier, axis=1)

tier_labels = {
    0: 'Never engaged',
    1: 'Early dropout (wk 4–7)',
    2: 'Mid dropout (wk 8–15)',
    3: 'Late dropout (wk 16–23)',
    4: 'Completed (wk 24)',
}
xrp_agg['retention_tier_label'] = xrp_agg['retention_tier'].map(tier_labels)

print(f"Shape: {xrp_agg.shape}")
print(f"Unique patients: {xrp_agg['PATID'].nunique()}")
print(xrp_agg.head())

Shape: (555, 7)
Unique patients: 555
   PATID  n_xrp_visits  last_xrp_week  had_relapse  first_relapse_week  \
0    323            13           19.0            1                19.0   
1   3424             4            7.0            1                 7.0   
2   3735            10           16.0            1                16.0   
3   3860             1            6.0            1                 6.0   
4   4138             1            7.0            1                 7.0   

   retention_tier     retention_tier_label  
0               3  Late dropout (wk 16–23)  
1               1   Early dropout (wk 4–7)  
2               3  Late dropout (wk 16–23)  
3               1   Early dropout (wk 4–7)  
4               1   Early dropout (wk 4–7)  


In [6]:
# ── Assessment ────────────────────────────────────────────────────────────────
print("── Tier distribution ────────────────────────────────────────────────────")
tier_summary = (
    xrp_agg.groupby(['retention_tier', 'retention_tier_label'])
    .agg(
        n              = ('PATID',             'count'),
        pct            = ('PATID',             lambda x: round(len(x) / len(xrp_agg) * 100, 1)),
        relapse_rate   = ('had_relapse',        'mean'),
        mean_last_week = ('last_xrp_week',      'mean'),
        mean_n_visits  = ('n_xrp_visits',       'mean'),
    )
    .round(2)
    .reset_index()
)
print(tier_summary.to_string(index=False))



── Tier distribution ────────────────────────────────────────────────────
 retention_tier    retention_tier_label   n  pct  relapse_rate  mean_last_week  mean_n_visits
              1  Early dropout (wk 4–7) 164 29.5          0.98            6.23           1.81
              2   Mid dropout (wk 8–15) 108 19.5          0.96           11.06           5.85
              3 Late dropout (wk 16–23)  61 11.0          0.87           19.44          13.44
              4       Completed (wk 24) 222 40.0          0.04           24.00          19.27


In [8]:
# ── Save retention tier as target variable file ───────────────────────────────
retention_tier = xrp_agg[['PATID', 'retention_tier']].copy()
retention_tier['completed'] = (retention_tier['retention_tier'] == 4).astype(int)

retention_tier.to_csv("../clean-data/retention_tier.csv", index=False)

print(f"Saved → retention_tier.csv")
print(f"Shape : {retention_tier.shape}")
print(f"\nTier distribution:")
print(retention_tier['retention_tier'].value_counts().sort_index().to_string())
print(f"\nCompleted distribution:")
print(retention_tier['completed'].value_counts().sort_index().to_string())

Saved → retention_tier.csv
Shape : (555, 3)

Tier distribution:
retention_tier
1    164
2    108
3     61
4    222

Completed distribution:
completed
0    333
1    222
